In [1]:
from data_fetcher import DataFetcher, DataFetcherConfig
from data_processor import OptionGammaProcessor, OptionGammaConfig
from utils import *
from pathlib import Path
from naive_exp import *

In [3]:
cfg = DataFetcherConfig(
        wrds_username="sniperw",
        data_dir="data",
        start_date="2018-01-01",
        end_date="2024-12-31",
        start_year=2018,
        end_year=2024,
        file_type="parquet",
        compression="zstd",
        replace=True,
        crsp_permno_chunk_size=500,
        optionm_secid_chunk_size=5,
        min_abs_prc=None,
        include_ticker_fallback=False,
        keep_intermediate_csv=False,
    )

"""cfg = DataFetcherConfig(
        wrds_username="sniperw",
        data_dir="data",
        start_date="2000-01-01",
        end_date="2025-12-31",
        start_year=2000,
        end_year=2025,
        file_type="parquet",
        compression="zstd",
        replace=False,
        crsp_permno_chunk_size=500,
        optionm_secid_chunk_size=25,
        min_abs_prc=5.0,
        include_ticker_fallback=False,
        keep_intermediate_csv=False,
    )"""
fetcher = DataFetcher(cfg)

In [3]:
fetcher.run_identifier_pipeline()

Loading library list...
Done


In [6]:
log_parquet_inventory(Path("data"))
log_parquet_inventory(Path("results/option_gamma"))

WindowsPath('results/option_gamma/parquet_inventory_log.txt')

In [3]:
fetcher.fetch_crsp_dsf(combine_final=True)

Loading library list...
Done


CRSP DSF month-chunks:   0%|          | 0/1152 [00:00<?, ?chunk/s]

Loading library list...
Loading library list...
Loading library list...
Loading library list...
Loading library list...
Done
Done
Done
Done
Done


Combine CRSP DSF months:   0%|          | 0/96 [00:00<?, ?month/s]

WindowsPath('data/crsp_daily_common_2018_2025.parquet')

In [4]:
fetcher.build_crsp_liquidity_panel()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WindowsPath('data/crsp_liquidity_panel_2018_2025.parquet')

In [4]:
fetcher.build_top_liquid_stock_universe(top_pct=0.01)

WindowsPath('data/crsp_top_liquid_universe_2018_2024.parquet')

In [5]:
fetcher.build_linked_secid_file_from_top_liquid()

WindowsPath('data/linked_secids_top_liquid_2018_2024.parquet')

In [8]:
fetcher.fetch_opprcd()

opprcd month-chunks:   0%|          | 0/336 [00:00<?, ?task/s]

Loading library list...
Loading library list...
Loading library list...
Loading library list...
Loading library list...
Done
Done
Done
Done
Done


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WindowsPath('data/opprcd_linked_2018_2024.parquet')

In [9]:
fetcher.fetch_secprd()

secprd month-chunks:   0%|          | 0/336 [00:00<?, ?task/s]

Loading library list...
Loading library list...
Loading library list...
Loading library list...
Loading library list...
Done
Done
Done
Done
Done


WindowsPath('data/secprd_linked_2018_2024.parquet')

In [10]:
export_parquet_sample_to_csv("data/opprcd_linked_2018_2024.parquet", n=2000, order_by="secid")

WindowsPath('data/opprcd_linked_2018_2024_head_2000.csv')

In [11]:
fetcher.close()

In [12]:
cfg = OptionGammaConfig(
    option_parquet="data/opprcd_linked_2018_2024.parquet",
    spot_parquet="data/secprd_linked_2018_2024.parquet",
    output_dir="results/option_gamma",
    start_date="2018-01-01",
    end_date="2024-12-31",
    secids=None,
    spot_price_field="close",
    require_spot=True,
    memory_limit="16GB",
    threads=8,
)

processor = OptionGammaProcessor(cfg)


[2026-03-20 02:46:33] [INFO] OptionGammaProcessor_2586746077552 - Registering option parquet: data\opprcd_linked_2018_2024.parquet
[2026-03-20 02:46:34] [INFO] OptionGammaProcessor_2586746077552 - Registering spot parquet: data\secprd_linked_2018_2024.parquet


In [13]:
diag = processor.run_diagnostics()
print(diag["spot_join_coverage_report"])

[2026-03-20 02:46:36] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 0.03s
[2026-03-20 02:46:37] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 0.79s
[2026-03-20 02:46:37] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 0.05s
[2026-03-20 02:46:38] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 0.73s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-03-20 02:46:41] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 3.34s
[2026-03-20 02:46:41] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 0.00s
[2026-03-20 02:46:41] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 0.01s
[2026-03-20 02:46:42] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 0.91s


   n_option_rows  n_matched_spot  n_missing_spot  matched_ratio
0       47580683      47580683.0             0.0            1.0


In [14]:
processor.run_baseline_pipeline(
    export_contract_gex="results/option_gamma/contract_gex.parquet",
    export_underlying_daily="results/option_gamma/underlying_gex_daily.parquet",
)

[2026-03-20 02:46:49] [INFO] OptionGammaProcessor_2586746077552 - Exporting query to parquet: results\option_gamma\contract_gex.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-03-20 02:47:11] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 22.09s
[2026-03-20 02:47:11] [INFO] OptionGammaProcessor_2586746077552 - Exporting query to parquet: results\option_gamma\underlying_gex_daily.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-03-20 02:47:18] [INFO] OptionGammaProcessor_2586746077552 - Query finished in 6.93s


{'contract_gex': WindowsPath('results/option_gamma/contract_gex.parquet'),
 'underlying_daily': WindowsPath('results/option_gamma/underlying_gex_daily.parquet')}

In [15]:
processor.close()

[2026-03-20 02:47:22] [INFO] OptionGammaProcessor_2586746077552 - Closing DuckDB connection


In [16]:
export_parquet_sample_to_csv("results/option_gamma/contract_gex.parquet", n=1000, order_by="date")
export_parquet_sample_to_csv("results/option_gamma/underlying_gex_daily.parquet", frac=1,order_by="date")

WindowsPath('results/option_gamma/underlying_gex_daily_sample_100pct.csv')

In [2]:
out = export_secid_crsp_name_mapping(
    secid_file=r"E:\Pythonfiles\ProjectGamma\data\linked_secids_top_liquid_2018_2024.parquet",
    stocknames_file=r"E:\Pythonfiles\ProjectGamma\data\crsp_common_stocknames_2018_2024.parquet",
    output_file=r"E:\Pythonfiles\ProjectGamma\data\secid_crsp_name_mapping_2018_2024.csv",
    replace=True,
    keep_all_name_rows=True,
)

In [2]:
underlying_path = r"E:\Pythonfiles\ProjectGamma\results\option_gamma\underlying_gex_daily.parquet"
contract_path = r"E:\Pythonfiles\ProjectGamma\results\option_gamma\contract_gex.parquet"
out_dir = r"E:\Pythonfiles\ProjectGamma\results\option_gamma\experiments\gamma_ranking"
store = ParquetDataStore(
        underlying_path=underlying_path,
        contract_path=contract_path,
    )

In [3]:
config_contract = GammaRankingExperimentConfig(
    name="gamma_rank_contract_filtered",
    mode="contract_aggregated",
    contract_aggregation=ContractAggregationSpec(
        agg_map={"gex_dollar_1pct": "sum"},
        keep_extra_metrics=True,
    ),
    return_horizon_days=1,
    n_buckets=5,
    weighting="equal",
    min_cross_section=10,
    date_start="2018-01-01",
    date_end="2024-12-31",
    factor_winsor_lower=0.01,
    factor_winsor_upper=0.99,
)

experiment_contract = GammaRankingEffectExperiment(
    store=store,
    config=config_contract,
    output_dir=out_dir,
    verbose=True,
)


underlying_filters = FilterSet(filters=[
    ColumnFilter(DATASET_UNDERLYING, "n_contracts", ">=", 10),
    ColumnFilter(DATASET_UNDERLYING, "spot_close", ">=", 2.0),
], label="u_loose")


contract_filters = FilterSet(filters=[
    ColumnFilter(DATASET_CONTRACT, "dte", "between", 0, 7),
    ColumnFilter(DATASET_CONTRACT, "spot_moneyness", "between", 0.90, 1.10),
    ColumnFilter(DATASET_CONTRACT, "open_interest", ">=", 10),
], label="atm_midcurve_oi50")

artifacts_contract = experiment_contract.run(
    underlying_filters=underlying_filters,
    contract_filters=contract_filters,
    save_outputs=True,
    run_label="contract_filtered_grid_best",
)

print("\n=== CONTRACT-FILTERED SUMMARY ===")
print(artifacts_contract.summary.T)

[DEBUG] underlying base rows after date window: 32,461
[DEBUG] underlying filter 1: underlying.n_contracts:>=(10) | rows 32,461 -> 32,426 | kept_vs_prev=99.89% | removed_vs_prev=0.11% | kept_vs_base=99.89% | removed_vs_base=0.11%
[DEBUG] underlying filter 2: underlying.spot_close:>=(2.0) | rows 32,426 -> 32,425 | kept_vs_prev=100.00% | removed_vs_prev=0.00% | kept_vs_base=99.89% | removed_vs_base=0.11%
[DEBUG] contract base rows after date window: 47,580,683
[DEBUG] contract filter 1: contract.dte:between(0,7) | rows 47,580,683 -> 3,568,418 | kept_vs_prev=7.50% | removed_vs_prev=92.50% | kept_vs_base=7.50% | removed_vs_base=92.50%
[DEBUG] contract filter 2: contract.spot_moneyness:between(0.9,1.1) | rows 3,568,418 -> 1,277,383 | kept_vs_prev=35.80% | removed_vs_prev=64.20% | kept_vs_base=2.68% | removed_vs_base=97.32%
[DEBUG] contract filter 3: contract.open_interest:>=(10) | rows 1,277,383 -> 1,215,951 | kept_vs_prev=95.19% | removed_vs_prev=4.81% | kept_vs_base=2.56% | removed_vs_bas